# Rebuild All Figures From Summarized Outputs

이 notebook은 `src` visualization helper를 import하지 않고, `summarized_outputs`/figure-input CSV만 읽어서 top-level `Complexity/Figures` 그림을 재생성합니다. 각 그림은 별도 code cell 하나에 대응합니다.


In [ ]:
from pathlib import Path
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

FIGURES_DIR = Path.cwd()
if FIGURES_DIR.name != "Figures":
    FIGURES_DIR = Path("/home/bjyong/Complexity/Complexity/Figures")
ROOT = FIGURES_DIR.parent
OUT = FIGURES_DIR / "notebook_regenerated"
OUT.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 220,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

def csv(relpath):
    path = ROOT / relpath
    if not path.exists():
        raise FileNotFoundError(path)
    return pd.read_csv(path)

def out(name):
    return OUT / name

def finish(fig, name):
    fig.tight_layout()
    path = out(name)
    fig.savefig(path)
    plt.close(fig)
    print(path)
    return path

def _label(value):
    if isinstance(value, float):
        return f"{value:g}"
    return str(value)

def line_plot(relpath, name, x, y, group=None, title="", xlabel="d", ylabel="", yerr=None, max_groups=None):
    df = csv(relpath).sort_values([group, x] if group and group in csv(relpath).columns else [x])
    fig, ax = plt.subplots(figsize=(7.2, 4.4))
    if group and group in df.columns:
        groups = list(df.groupby(group, sort=True))
        if max_groups:
            groups = groups[:max_groups]
        for key, sub in groups:
            sub = sub.sort_values(x)
            if yerr and yerr in sub.columns:
                ax.errorbar(sub[x], sub[y], yerr=sub[yerr], linewidth=1.4, capsize=2, label=f"{group}={_label(key)}")
            else:
                ax.plot(sub[x], sub[y], linewidth=1.5, label=f"{group}={_label(key)}")
        ax.legend(fontsize=7, ncol=2)
    else:
        df = df.sort_values(x)
        if yerr and yerr in df.columns:
            ax.errorbar(df[x], df[y], yerr=df[yerr], linewidth=1.6, capsize=2)
        else:
            ax.plot(df[x], df[y], linewidth=1.8)
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel or y)
    return finish(fig, name)

def scatter_or_bar(relpath, name, x, y, title="", xlabel="", ylabel="", yerr=None, hue=None, connect=True):
    df = csv(relpath)
    fig, ax = plt.subplots(figsize=(6.4, 4.2))
    if hue and hue in df.columns:
        for key, sub in df.groupby(hue, sort=True):
            sub = sub.sort_values(x)
            ax.errorbar(sub[x], sub[y], yerr=sub[yerr] if yerr and yerr in sub.columns else None, marker="o", linewidth=1.3 if connect else 0, capsize=2, label=f"{hue}={_label(key)}")
        ax.legend(fontsize=7)
    else:
        sub = df.sort_values(x) if x in df.columns else df
        ax.errorbar(sub[x], sub[y], yerr=sub[yerr] if yerr and yerr in sub.columns else None, marker="o", linewidth=1.4 if connect else 0, capsize=2)
    ax.set_title(title)
    ax.set_xlabel(xlabel or x)
    ax.set_ylabel(ylabel or y)
    return finish(fig, name)

def two_panel_phase(curve_relpath, phase_relpath, name, curve_group, curve_y, phase_x, title="", phase_y="A_kappa_mean", curve_yerr=None, phase_yerr="A_kappa_sem"):
    curves = csv(curve_relpath)
    phase = csv(phase_relpath)
    fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(10.0, 4.1))
    for key, sub in curves.groupby(curve_group, sort=True):
        sub = sub.sort_values("radius")
        ax0.plot(sub["radius"], sub[curve_y], linewidth=1.35, label=f"{curve_group}={_label(key)}")
    ax0.set_xlabel("d")
    ax0.set_ylabel(curve_y)
    ax0.legend(fontsize=7)
    phase = phase.sort_values(phase_x)
    ax1.errorbar(phase[phase_x], phase[phase_y], yerr=phase[phase_yerr] if phase_yerr in phase.columns else None, marker="o", linewidth=1.5, capsize=2)
    ax1.set_xlabel(phase_x)
    ax1.set_ylabel(phase_y)
    fig.suptitle(title)
    return finish(fig, name)

def qc_plot(relpath, name, x, y, group=None, title="", threshold_col=None, claim_col=None):
    df = csv(relpath)
    fig, ax = plt.subplots(figsize=(7.0, 4.3))
    if group and group in df.columns:
        for key, sub in df.groupby(group, sort=True):
            sub = sub.sort_values(x)
            ax.plot(sub[x], sub[y], marker="o", markersize=3, linewidth=1.2, label=f"{group}={_label(key)}")
        ax.legend(fontsize=7, ncol=2)
    else:
        df = df.sort_values(x)
        ax.plot(df[x], df[y], marker="o", markersize=3, linewidth=1.2)
    if threshold_col and threshold_col in df.columns:
        ax.axhline(float(df[threshold_col].dropna().iloc[0]), color="black", linestyle="--", linewidth=1.0, label="threshold")
    if claim_col and claim_col in df.columns:
        counts = df[claim_col].value_counts().to_dict()
        ax.text(0.01, 0.98, str(counts), transform=ax.transAxes, va="top", fontsize=8)
    ax.set_title(title)
    ax.set_xlabel(x)
    ax.set_ylabel(y)
    return finish(fig, name)

def theory_analytic():
    df = csv("01_theory/01_theory_analytic/summarized_outputs/fig01_phi_by_analytic_solution_alpha0p1.csv").sort_values("r")
    y = df["phi_rel"] if "phi_rel" in df else df["phi"] - df["phi"].iloc[0]
    fig, ax = plt.subplots(figsize=(7.2, 4.4))
    ax.plot(df["r"], y, color="black", linewidth=2)
    ax.set_title("Analytic full-RS phi(d), alpha=0.1")
    ax.set_xlabel("d")
    ax.set_ylabel("phi(d) - phi(d0)")
    return finish(fig, "fig01_theory_analytic_phi_by_distance_alpha0p1.png")

def theory_sampling():
    df = csv("01_theory/02_theory_sampling/summarized_outputs/fig01_sampling_phi_by_distance.csv")
    fig, ax = plt.subplots(figsize=(7.2, 4.4))
    for n, sub in df.groupby("N", sort=True):
        sub = sub.sort_values("r")
        y = sub["phi_emp_rel"] if "phi_emp_rel" in sub else sub["phi_emp"] - sub["phi_emp"].iloc[0]
        ax.plot(sub["r"], y, marker="o", markersize=3, linewidth=1.4, label=f"N={int(n)}")
    ax.legend(fontsize=8)
    ax.set_title("Two-pool shell sampling, alpha=0.1")
    ax.set_xlabel("d")
    ax.set_ylabel("empirical phi(d) - phi(d0)")
    return finish(fig, "fig02_theory_sampling_phi_by_distance_alpha0p1.png")

def theory_combined():
    analytic = csv("01_theory/01_theory_analytic/summarized_outputs/fig01_phi_by_analytic_solution_alpha0p1.csv").sort_values("r")
    sampling = csv("01_theory/02_theory_sampling/summarized_outputs/fig01_sampling_phi_by_distance.csv")
    fig, ax = plt.subplots(figsize=(8.0, 4.8))
    ay = analytic["phi_rel"] if "phi_rel" in analytic else analytic["phi"] - analytic["phi"].iloc[0]
    ax.plot(analytic["r"], ay, color="black", linewidth=2.2, label="analytic")
    for n, sub in sampling.groupby("N", sort=True):
        sub = sub.sort_values("r")
        y = sub["phi_emp_rel"] if "phi_emp_rel" in sub else sub["phi_emp"] - sub["phi_emp"].iloc[0]
        ax.plot(sub["r"], y, marker="o", markersize=2.8, linewidth=1.3, label=f"N={int(n)}")
    ax.legend(fontsize=7)
    ax.set_title("Analytic vs shell sampling, alpha=0.1")
    ax.set_xlabel("d")
    ax.set_ylabel("phi(d) - phi(d0)")
    return finish(fig, "fig03_theory_sampling_vs_analytic_phi_by_distance_alpha0p1.png")

def random_dataset_preview():
    points = csv("02_dnn_synthetic/06_random_baseline/01_dataset/summarized_outputs/gaussian_random_90_dataset/example_dataset_points.csv")
    fig, ax = plt.subplots(figsize=(5.2, 5.0))
    ax.scatter(points["x0"], points["x1"], c=points["label"], s=18, cmap="coolwarm", alpha=0.85, edgecolors="none")
    ax.set_title("Gaussian random baseline dataset example")
    ax.set_xlabel("x0")
    ax.set_ylabel("x1")
    ax.set_aspect("equal", adjustable="box")
    return finish(fig, "dnn_synthetic_random_baseline_dataset_example.png")


## Theory analytic


In [ ]:
theory_analytic()


## Theory sampling


In [ ]:
theory_sampling()


## Theory combined


In [ ]:
theory_combined()


## DNN synthetic phi(d)


In [ ]:
line_plot('02_dnn_synthetic/05_proxy_local_entropy/summarized_outputs/18_beta_cell_90_dataset_30_reference/d_0.01_to_2.50_dense/summary_tables/absolute_phi_by_beta_radius.csv', 'dnn_synthetic_phi_d_curve.png', 'radius', 'phi_full', group='beta', title='DNN synthetic phi(d)', ylabel='phi_full')


## DNN synthetic energetic phi(d)


In [ ]:
line_plot('02_dnn_synthetic/05_proxy_local_entropy/summarized_outputs/18_beta_cell_90_dataset_30_reference/d_0.01_to_2.50_dense/summary_tables/absolute_phi_by_beta_radius.csv', 'dnn_synthetic_phi_energetic_d_curve.png', 'radius', 'phi_energy', group='beta', title='DNN synthetic energetic phi(d)', ylabel='phi_energy')


## DNN synthetic derivative phi(d)


In [ ]:
line_plot('02_dnn_synthetic/05_proxy_local_entropy/summarized_outputs/18_beta_cell_90_dataset_30_reference/d_0.01_to_2.50_dense/summary_tables/dphi_dr_by_beta_radius.csv', 'dnn_synthetic_derivative_phi_d_curve.png', 'radius', 'dphi_full_dr', group='beta', title='DNN synthetic d phi / dd', ylabel='dphi_full_dr')


## DNN synthetic derivative energetic phi(d)


In [ ]:
line_plot('02_dnn_synthetic/05_proxy_local_entropy/summarized_outputs/18_beta_cell_90_dataset_30_reference/d_0.01_to_2.50_dense/summary_tables/dphi_dr_by_beta_radius.csv', 'dnn_synthetic_derivative_phi_energetic_d_curve.png', 'radius', 'dphi_energy_dr', group='beta', title='DNN synthetic energetic derivative', ylabel='dphi_energy_dr')


## DNN synthetic A by beta


In [ ]:
scatter_or_bar('02_dnn_synthetic/05_proxy_local_entropy/summarized_outputs/18_beta_cell_90_dataset_30_reference/d_0.01_to_2.50_dense/summary_tables/phase_like_A_measure.csv', 'dnn_synthetic_phase_like_A_by_beta.png', 'beta', 'A_transition_total_variation', title='A-measure by beta', xlabel='beta', ylabel='A')


## DNN synthetic A by complexity


In [ ]:
scatter_or_bar('02_dnn_synthetic/05_proxy_local_entropy/summarized_outputs/18_beta_cell_90_dataset_30_reference/d_0.01_to_2.50_dense/summary_tables/phase_like_A_measure.csv', 'dnn_synthetic_phase_like_A_by_complexity.png', 'complexity_mean', 'A_transition_total_variation', title='A-measure by complexity', xlabel='complexity', ylabel='A')


## DNN synthetic logZ split QC


In [ ]:
qc_plot('02_dnn_synthetic/05_proxy_local_entropy/summarized_outputs/qc/logZ_split_qc_results.csv', 'dnn_synthetic_logZ_split_qc_results.png', 'radius', 'max_split_logZ_per_P_diff', group='beta', title='DNN synthetic logZ split QC', threshold_col='threshold_max_split_logZ_per_P_diff', claim_col='claim')


## DNN synthetic reference variability


In [ ]:
line_plot('02_dnn_synthetic/05_proxy_local_entropy/summarized_outputs/qc/reference_variability_results.csv', 'dnn_synthetic_reference_variability_results.png', 'radius', 'mean_ref_se_logZ_inf_full', group='beta', title='DNN synthetic reference SE(logZ)', ylabel='mean ref SE logZ')


## DNN synthetic dataset variability


In [ ]:
line_plot('02_dnn_synthetic/05_proxy_local_entropy/summarized_outputs/qc/dataset_variability_results.csv', 'dnn_synthetic_dataset_variability_results.png', 'radius', 'dataset_se_logZ_inf_full', group='beta', title='DNN synthetic dataset SE(logZ)', ylabel='dataset SE logZ')


## Random baseline dataset example


In [ ]:
random_dataset_preview()


## Random baseline complexity summary


In [ ]:
scatter_or_bar('02_dnn_synthetic/06_random_baseline/02_complexity_measure/summarized_outputs/gaussian_random_90_dataset_30_reference/summary_tables/spin_beta_curve_with_random_baseline_marker.csv', 'dnn_synthetic_random_baseline_complexity_summary.png', 'beta', 'complexity_mean', title='Spin beta complexity with random baseline', xlabel='spin beta', ylabel='complexity');
point = csv('02_dnn_synthetic/06_random_baseline/02_complexity_measure/summarized_outputs/gaussian_random_90_dataset_30_reference/summary_tables/random_baseline_complexity_point.csv').iloc[0]; print('random baseline complexity', point['complexity_mean'])


## Random baseline phi(d)


In [ ]:
line_plot('02_dnn_synthetic/06_random_baseline/05_proxy_local_entropy/summarized_outputs/gaussian_random_90_dataset_30_reference/d_0.01_to_2.50_dense/summary_tables/absolute_phi_by_beta_radius.csv', 'dnn_synthetic_random_baseline_phi_d_curve.png', 'radius', 'phi_full', group='beta', title='Random baseline phi(d)', ylabel='phi_full')


## Random baseline energetic phi(d)


In [ ]:
line_plot('02_dnn_synthetic/06_random_baseline/05_proxy_local_entropy/summarized_outputs/gaussian_random_90_dataset_30_reference/d_0.01_to_2.50_dense/summary_tables/absolute_phi_by_beta_radius.csv', 'dnn_synthetic_random_baseline_phi_energetic_d_curve.png', 'radius', 'phi_energy', group='beta', title='Random baseline energetic phi(d)', ylabel='phi_energy')


## Random baseline derivative phi(d)


In [ ]:
line_plot('02_dnn_synthetic/06_random_baseline/05_proxy_local_entropy/summarized_outputs/gaussian_random_90_dataset_30_reference/d_0.01_to_2.50_dense/summary_tables/dphi_dr_by_beta_radius.csv', 'dnn_synthetic_random_baseline_derivative_phi_d_curve.png', 'radius', 'dphi_full_dr', group='beta', title='Random baseline derivative phi(d)', ylabel='dphi_full_dr')


## Random baseline derivative energetic phi(d)


In [ ]:
line_plot('02_dnn_synthetic/06_random_baseline/05_proxy_local_entropy/summarized_outputs/gaussian_random_90_dataset_30_reference/d_0.01_to_2.50_dense/summary_tables/dphi_dr_by_beta_radius.csv', 'dnn_synthetic_random_baseline_derivative_phi_energetic_d_curve.png', 'radius', 'dphi_energy_dr', group='beta', title='Random baseline derivative energetic phi(d)', ylabel='dphi_energy_dr')


## Random baseline A by beta


In [ ]:
scatter_or_bar('02_dnn_synthetic/06_random_baseline/05_proxy_local_entropy/summarized_outputs/gaussian_random_90_dataset_30_reference/d_0.01_to_2.50_dense/summary_tables/phase_like_A_measure.csv', 'dnn_synthetic_random_baseline_phase_like_A_by_beta.png', 'beta', 'A_transition_total_variation', title='Random baseline A by beta', xlabel='source beta', ylabel='A')


## Random baseline A by complexity


In [ ]:
scatter_or_bar('02_dnn_synthetic/06_random_baseline/05_proxy_local_entropy/summarized_outputs/gaussian_random_90_dataset_30_reference/d_0.01_to_2.50_dense/summary_tables/phase_like_A_measure.csv', 'dnn_synthetic_random_baseline_phase_like_A_by_complexity.png', 'complexity_mean', 'A_transition_total_variation', title='Random baseline A by complexity', xlabel='complexity', ylabel='A')


## Random baseline logZ split QC


In [ ]:
qc_plot('02_dnn_synthetic/06_random_baseline/05_proxy_local_entropy/summarized_outputs/qc/logZ_split_qc_results.csv', 'dnn_synthetic_random_baseline_logZ_split_qc_results.png', 'radius', 'max_split_logZ_per_P_diff', group='beta', title='Random baseline logZ split QC', threshold_col='threshold_max_split_logZ_per_P_diff', claim_col='claim')


## Random baseline reference variability


In [ ]:
line_plot('02_dnn_synthetic/06_random_baseline/05_proxy_local_entropy/summarized_outputs/qc/reference_variability_results.csv', 'dnn_synthetic_random_baseline_reference_variability_results.png', 'radius', 'mean_ref_se_logZ_inf_full', group='beta', title='Random baseline reference SE(logZ)', ylabel='mean ref SE logZ')


## Random baseline dataset variability


In [ ]:
line_plot('02_dnn_synthetic/06_random_baseline/05_proxy_local_entropy/summarized_outputs/qc/dataset_variability_results.csv', 'dnn_synthetic_random_baseline_dataset_variability_results.png', 'radius', 'dataset_se_logZ_inf_full', group='beta', title='Random baseline dataset SE(logZ)', ylabel='dataset SE logZ')


## Synthetic plus random phi(d)


In [ ]:
line_plot('02_dnn_synthetic/05_proxy_local_entropy/summarized_outputs/18_beta_cell_90_dataset_30_reference/d_0.01_to_2.50_dense/summary_tables/absolute_phi_by_beta_radius.csv', 'dnn_synthetic_with_random_baseline_phi_d_curve.png', 'radius', 'phi_full', group='beta', title='Spin phi(d); compare with random baseline separately', ylabel='phi_full')


## Synthetic plus random energetic phi(d)


In [ ]:
line_plot('02_dnn_synthetic/05_proxy_local_entropy/summarized_outputs/18_beta_cell_90_dataset_30_reference/d_0.01_to_2.50_dense/summary_tables/absolute_phi_by_beta_radius.csv', 'dnn_synthetic_with_random_baseline_phi_energetic_d_curve.png', 'radius', 'phi_energy', group='beta', title='Spin energetic phi(d); random baseline has own panel', ylabel='phi_energy')


## Synthetic plus random derivative phi(d)


In [ ]:
line_plot('02_dnn_synthetic/05_proxy_local_entropy/summarized_outputs/18_beta_cell_90_dataset_30_reference/d_0.01_to_2.50_dense/summary_tables/dphi_dr_by_beta_radius.csv', 'dnn_synthetic_with_random_baseline_derivative_phi_d_curve.png', 'radius', 'dphi_full_dr', group='beta', title='Spin derivative phi(d); random baseline has own panel', ylabel='dphi_full_dr')


## Synthetic plus random derivative energetic phi(d)


In [ ]:
line_plot('02_dnn_synthetic/05_proxy_local_entropy/summarized_outputs/18_beta_cell_90_dataset_30_reference/d_0.01_to_2.50_dense/summary_tables/dphi_dr_by_beta_radius.csv', 'dnn_synthetic_with_random_baseline_derivative_phi_energetic_d_curve.png', 'radius', 'dphi_energy_dr', group='beta', title='Spin energetic derivative; random baseline has own panel', ylabel='dphi_energy_dr')


## Synthetic plus random A by beta


In [ ]:
scatter_or_bar('02_dnn_synthetic/06_random_baseline/05_proxy_local_entropy/summarized_outputs/with_random_baseline/phase_like_A_with_random_baseline.csv', 'dnn_synthetic_with_random_baseline_phase_like_A_by_beta.png', 'beta', 'A_transition_total_variation', title='A by beta with random baseline row', xlabel='beta', ylabel='A', hue='series')


## Synthetic plus random A by complexity


In [ ]:
scatter_or_bar('02_dnn_synthetic/06_random_baseline/05_proxy_local_entropy/summarized_outputs/with_random_baseline/phase_like_A_with_random_baseline.csv', 'dnn_synthetic_with_random_baseline_phase_like_A_by_complexity.png', 'complexity_mean', 'A_transition_total_variation', title='A by complexity with random baseline row', xlabel='complexity', ylabel='A', hue='series')


## MNIST label dataset eta sweep


In [ ]:
scatter_or_bar('03_dnn_mnist/label_noise_sweep/01_dataset/summarized_outputs/eta_dataset_index.csv', 'dnn_mnist_label_noise_dataset_eta_sweep.png', 'eta', 'n_train', title='Label-noise datasets by eta', xlabel='eta', ylabel='n_train')


## MNIST label complexity by eta


In [ ]:
scatter_or_bar('03_dnn_mnist/label_noise_sweep/02_complexity_measure/summarized_outputs/complexity_axis_spin_mnist_30ref_eta0p02_0p05_0p15_0p25/label_noise_complexity_by_eta.csv', 'dnn_mnist_label_noise_complexity_by_eta.png', 'eta', 'complexity_proxy', title='Label-noise complexity by eta', xlabel='eta', ylabel='complexity')


## MNIST label phi(d)


In [ ]:
line_plot('03_dnn_mnist/label_noise_sweep/05_proxy_local_entropy/summarized_outputs/figure_inputs/phi_d_curve/phi_d_curve.csv', 'dnn_mnist_label_noise_phi_d_curve.png', 'radius', 'delta_phi_energy_mean', group='eta', title='MNIST label-noise phi(d)', ylabel='delta phi energy')


## MNIST label energetic phi(d)


In [ ]:
line_plot('03_dnn_mnist/label_noise_sweep/05_proxy_local_entropy/summarized_outputs/figure_inputs/phi_energetic_d_curve/phi_energetic_d_curve.csv', 'dnn_mnist_label_noise_phi_energetic_d_curve.png', 'radius', 'phi_energy_raw_mean', group='eta', title='MNIST label-noise energetic phi(d)', ylabel='phi energy raw')


## MNIST label derivative phi(d)


In [ ]:
line_plot('03_dnn_mnist/label_noise_sweep/05_proxy_local_entropy/summarized_outputs/figure_inputs/derivative_phi_d_curve/derivative_phi_d_curve.csv', 'dnn_mnist_label_noise_derivative_phi_d_curve.png', 'radius', 'd_phi_energy_direct_dd', group='eta', title='MNIST label direct derivative phi(d)', ylabel='direct dphi/dd', yerr='d_phi_energy_direct_dd_sem')


## MNIST label derivative energetic phi(d)


In [ ]:
line_plot('03_dnn_mnist/label_noise_sweep/05_proxy_local_entropy/summarized_outputs/figure_inputs/derivative_phi_energetic_d_curve/derivative_phi_energetic_d_curve.csv', 'dnn_mnist_label_noise_derivative_phi_energetic_d_curve.png', 'radius', 'd_phi_energy_direct_dd', group='eta', title='MNIST label direct energetic derivative', ylabel='direct dphi/dd', yerr='d_phi_energy_direct_dd_sem')


## MNIST label phase A by eta


In [ ]:
two_panel_phase('03_dnn_mnist/label_noise_sweep/05_proxy_local_entropy/summarized_outputs/figure_inputs/phase_like_A_by_eta/phase_derivative_curves.csv', '03_dnn_mnist/label_noise_sweep/05_proxy_local_entropy/summarized_outputs/figure_inputs/phase_like_A_by_eta/phase_like_A_by_eta.csv', 'dnn_mnist_label_noise_phase_like_A_by_eta.png', 'eta', 'dphi_dr_smooth_mean', 'eta', title='MNIST label direct derivative and A by eta')


## MNIST label phase A by complexity


In [ ]:
two_panel_phase('03_dnn_mnist/label_noise_sweep/05_proxy_local_entropy/summarized_outputs/figure_inputs/phase_like_A_by_complexity/phase_derivative_curves.csv', '03_dnn_mnist/label_noise_sweep/05_proxy_local_entropy/summarized_outputs/figure_inputs/phase_like_A_by_complexity/phase_like_A_by_complexity.csv', 'dnn_mnist_label_noise_phase_like_A_by_complexity.png', 'eta', 'dphi_dr_smooth_mean', 'nmstv', title='MNIST label direct derivative and A by complexity')


## MNIST label logZ split QC


In [ ]:
qc_plot('03_dnn_mnist/label_noise_sweep/05_proxy_local_entropy/summarized_outputs/figure_inputs/logZ_split_qc_results/logZ_split_qc_results.csv', 'dnn_mnist_label_noise_logZ_split_qc_results.png', 'radius', 'max_split_logZ_per_P_diff', group='eta', title='MNIST label logZ split QC', threshold_col='threshold_max_split_logZ_per_P_diff', claim_col='claim')


## MNIST label reference variability


In [ ]:
line_plot('03_dnn_mnist/label_noise_sweep/05_proxy_local_entropy/summarized_outputs/figure_inputs/reference_variability_results/reference_variability_results.csv', 'dnn_mnist_label_noise_reference_variability_results.png', 'radius', 'reference_se_logZ_inf_full', group='eta', title='MNIST label reference SE(logZ)', ylabel='reference SE logZ')


## MNIST manual phi(d)


In [ ]:
line_plot('03_dnn_mnist/manual_rules/05_proxy_local_entropy/summarized_outputs/figure_inputs/phi_d_curve/phi_d_curve.csv', 'dnn_mnist_manual_rules_phi_d_curve.png', 'radius', 'delta_phi_energy_unit_mean', group='rule_label', title='MNIST manual-rules phi(d)', ylabel='delta phi energy', yerr='delta_phi_energy_unit_sem')


## MNIST manual energetic phi(d)


In [ ]:
line_plot('03_dnn_mnist/manual_rules/05_proxy_local_entropy/summarized_outputs/figure_inputs/phi_energetic_d_curve/phi_energetic_d_curve.csv', 'dnn_mnist_manual_rules_phi_energetic_d_curve.png', 'radius', 'phi_energy_raw_mean', group='rule_label', title='MNIST manual-rules energetic phi(d)', ylabel='phi energy raw', yerr='phi_energy_raw_sem')


## MNIST manual derivative phi(d)


In [ ]:
line_plot('03_dnn_mnist/manual_rules/05_proxy_local_entropy/summarized_outputs/figure_inputs/derivative_phi_d_curve/derivative_phi_d_curve.csv', 'dnn_mnist_manual_rules_derivative_phi_d_curve.png', 'radius', 'd_phi_energy_direct_dd_unit_mean', group='rule_label', title='MNIST manual-rules direct derivative phi(d)', ylabel='direct dphi/dd', yerr='d_phi_energy_direct_dd_unit_sem')


## MNIST manual derivative energetic phi(d)


In [ ]:
line_plot('03_dnn_mnist/manual_rules/05_proxy_local_entropy/summarized_outputs/figure_inputs/derivative_phi_energetic_d_curve/derivative_phi_energetic_d_curve.csv', 'dnn_mnist_manual_rules_derivative_phi_energetic_d_curve.png', 'radius', 'd_phi_energy_direct_dd_unit_mean', group='rule_label', title='MNIST manual-rules direct energetic derivative', ylabel='direct dphi/dd', yerr='d_phi_energy_direct_dd_unit_sem')


## MNIST manual phase A by rule


In [ ]:
two_panel_phase('03_dnn_mnist/manual_rules/05_proxy_local_entropy/summarized_outputs/figure_inputs/phase_like_A_by_rule/phase_derivative_curves.csv', '03_dnn_mnist/manual_rules/05_proxy_local_entropy/summarized_outputs/figure_inputs/phase_like_A_by_rule/phase_like_A_by_rule.csv', 'dnn_mnist_manual_rules_phase_like_A_by_rule.png', 'rule_label', 'dphi_dr_smooth_mean', 'rule_order', title='MNIST manual direct derivative and A by rule')


## MNIST manual phase A by complexity


In [ ]:
two_panel_phase('03_dnn_mnist/manual_rules/05_proxy_local_entropy/summarized_outputs/figure_inputs/phase_like_A_by_complexity/phase_derivative_curves.csv', '03_dnn_mnist/manual_rules/05_proxy_local_entropy/summarized_outputs/figure_inputs/phase_like_A_by_complexity/phase_like_A_by_complexity.csv', 'dnn_mnist_manual_rules_phase_like_A_by_complexity.png', 'rule_label', 'dphi_dr_smooth_mean', 'nmstv_mean', title='MNIST manual direct derivative and A by complexity')


## MNIST manual logZ split QC


In [ ]:
qc_plot('03_dnn_mnist/manual_rules/05_proxy_local_entropy/summarized_outputs/figure_inputs/logZ_split_qc_results/logZ_split_qc_results.csv', 'dnn_mnist_manual_rules_logZ_split_qc_results.png', 'radius', 'max_split_logZ_per_P_diff', group='rule_label', title='MNIST manual logZ split QC', threshold_col='threshold_max_split_logZ_per_P_diff', claim_col='qc_claim')


## MNIST manual reference variability


In [ ]:
line_plot('03_dnn_mnist/manual_rules/05_proxy_local_entropy/summarized_outputs/figure_inputs/reference_variability_results/reference_variability_results.csv', 'dnn_mnist_manual_rules_reference_variability_results.png', 'radius', 'reference_se_logZ_inf_full', group='rule_label', title='MNIST manual reference SE(logZ)', ylabel='reference SE logZ')
